# Carga de CSV

## Importar Librerías

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

## Descarga de Datos

In [2]:
# Definir los símbolos de las acciones
bbva_ticker = "BBVA.MC"   # BBVA (Bolsa de Madrid)
santander_ticker = "SAN.MC"  # Banco Santander (Bolsa de Madrid)

# Definir el rango de fechas
start_date = "2000-01-01"
end_date   = "2025-11-10"

# Descargar los datos históricos desde Yahoo Finance
bbva_data = yf.download(bbva_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)
santander_data = yf.download(santander_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)

# Mostrar resumen por consola
print("BBVA data:")
print(bbva_data.head())
print("\nSantander data:")
print(santander_data.head())

# Guardar los datos en CSV
bbva_data.to_csv("../csv/bbva_data.csv")
santander_data.to_csv("../csv/santander_data.csv")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

BBVA data:
Price      Adj Close      Close Dividends       High        Low       Open  \
Ticker       BBVA.MC    BBVA.MC   BBVA.MC    BBVA.MC    BBVA.MC    BBVA.MC   
Date                                                                         
2000-01-03  4.040036  13.623349       0.0  13.757854  13.594527  13.690602   
2000-01-04  3.934616  13.267874       0.0  13.536882  13.219837  13.450416   
2000-01-05  3.846295  12.970044       0.0  13.210230  12.912399  13.142977   
2000-01-06  3.846295  12.970044       0.0  12.970044  12.970044  12.970044   
2000-01-07  3.894730  13.133370       0.0  13.248659  12.998866  13.248659   

Price      Stock Splits    Volume  
Ticker          BBVA.MC   BBVA.MC  
Date                               
2000-01-03          0.0   8244257  
2000-01-04          0.0   8522096  
2000-01-05          0.0  12159826  
2000-01-06          0.0         0  
2000-01-07          0.0  62261944  

Santander data:
Price      Adj Close     Close Dividends      High       Lo

In [3]:
import yfinance as yf
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ===========================
# Tickers relacionados
# ===========================
santander_ticker = "SAN.MC"
bbva_ticker = "BBVA.MC"
ibex_ticker = "^IBEX"
eurusd_ticker = "EURUSD=X"
sp500_ticker = "^GSPC"
oil_ticker = "CL=F"

start_date = "2000-01-01"
end_date   = "2025-11-01"

# ===========================
# Descargar datos con columna 'Close'
# ===========================
data_san   = yf.download(santander_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Adj Close", "Close", "Volume", "High", "Low", "Open"]]
data_bbva  = yf.download(bbva_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_ibex  = yf.download(ibex_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_sp500 = yf.download(sp500_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_eurusd = yf.download(eurusd_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_oil   = yf.download(oil_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]

# ===========================
# Renombrar columnas
# ===========================
data_san.rename(columns={"Adj Close": "SAN_Adj_Close", "Close": "SAN_Close", "Volume": "SAN_Volume", "High": "SAN_High", "Low": "SAN_Low", "Open": "SAN_Open"}, inplace=True)
data_bbva.rename(columns={"Close": "BBVA_Price"}, inplace=True)
data_ibex.rename(columns={"Close": "IBEX"}, inplace=True)
data_sp500.rename(columns={"Close": "SP500"}, inplace=True)
data_eurusd.rename(columns={"Close": "EURUSD"}, inplace=True)
data_oil.rename(columns={"Close": "OIL"}, inplace=True)

# ===========================
# Unir todo por fecha
# ===========================
df_full = data_san.join([data_bbva, data_ibex, data_sp500, data_eurusd, data_oil], how="inner")
df_full.dropna(inplace=True)

print("✅ Dataset multivariable creado con columnas:")
print(df_full.columns.tolist())


# ===========================
# Guardar CSV normalizado
# ===========================
df_full.to_csv("../csv/santander_enriched.csv")
print("💾 Archivo guardado: 'csv/santander_enriched_scaled.csv'")

# Mostrar ejemplo
print(df_full.head())


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

✅ Dataset multivariable creado con columnas:
[('SAN_Adj_Close', 'SAN.MC'), ('SAN_Close', 'SAN.MC'), ('SAN_Volume', 'SAN.MC'), ('SAN_High', 'SAN.MC'), ('SAN_Low', 'SAN.MC'), ('SAN_Open', 'SAN.MC'), ('BBVA_Price', 'BBVA.MC'), ('IBEX', '^IBEX'), ('SP500', '^GSPC'), ('EURUSD', 'EURUSD=X'), ('OIL', 'CL=F')]
💾 Archivo guardado: 'csv/santander_enriched_scaled.csv'
Price      SAN_Adj_Close SAN_Close SAN_Volume  SAN_High   SAN_Low  SAN_Open  \
Ticker            SAN.MC    SAN.MC     SAN.MC    SAN.MC    SAN.MC    SAN.MC   
Date                                                                          
2003-12-01      2.415459  7.746234   42034006  7.772199  7.538513  7.746234   
2003-12-02      2.404664  7.711614  187525101  7.763544  7.633719  7.711614   
2003-12-03      2.420857  7.763544   31459710  7.763544  7.651029  7.763544   
2003-12-04      2.410062  7.728924   37904447  7.746234  7.668339  7.728924   
2003-12-05      2.401966  7.702959   20656590  7.728924  7.651029  7.702959   

Price  

In [4]:
# ===========================
# Tickers relacionados
# ===========================
santander_ticker = "SAN.MC"
bbva_ticker = "BBVA.MC"
ibex_ticker = "^IBEX"
eurusd_ticker = "EURUSD=X"
sp500_ticker = "^GSPC"
oil_ticker = "CL=F"

start_date = "2000-01-01"
end_date   = "2025-11-01"

# ===========================
# Descargar datos con columna 'Close'
# ===========================
data_bbva   = yf.download(bbva_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Adj Close", "Close", "Volume", "High", "Low", "Open"]]
data_san   = yf.download(santander_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_ibex  = yf.download(ibex_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_sp500 = yf.download(sp500_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_eurusd = yf.download(eurusd_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]
data_oil   = yf.download(oil_ticker, start=start_date, end=end_date, auto_adjust=False, actions=True)[["Close"]]

# ===========================
# Renombrar columnas
# ===========================
data_bbva.rename(columns={"Adj Close": "BBVA_Adj_Close", "Close": "BBVA_Close", "Volume": "BBVA_Volume", "High": "BBVA_High", "Low": "BBVA_Low", "Open": "BBVA_Open"}, inplace=True)
data_san.rename(columns={"Close": "SAN_Price"}, inplace=True)
data_ibex.rename(columns={"Close": "IBEX"}, inplace=True)
data_sp500.rename(columns={"Close": "SP500"}, inplace=True)
data_eurusd.rename(columns={"Close": "EURUSD"}, inplace=True)
data_oil.rename(columns={"Close": "OIL"}, inplace=True)

# ===========================
# Unir todo por fecha
# ===========================
df_full = data_san.join([data_bbva, data_ibex, data_sp500, data_eurusd, data_oil], how="inner")
df_full.dropna(inplace=True)

print("✅ Dataset multivariable creado con columnas:")
print(df_full.columns.tolist())


# ===========================
# Guardar CSV normalizado
# ===========================
df_full.to_csv("../csv/bbva_enriched.csv")
print("💾 Archivo guardado: 'csv/bbva_enriched.csv'")

# Mostrar ejemplo
print(df_full.head())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


✅ Dataset multivariable creado con columnas:
[('SAN_Price', 'SAN.MC'), ('BBVA_Adj_Close', 'BBVA.MC'), ('BBVA_Close', 'BBVA.MC'), ('BBVA_Volume', 'BBVA.MC'), ('BBVA_High', 'BBVA.MC'), ('BBVA_Low', 'BBVA.MC'), ('BBVA_Open', 'BBVA.MC'), ('IBEX', '^IBEX'), ('SP500', '^GSPC'), ('EURUSD', 'EURUSD=X'), ('OIL', 'CL=F')]
💾 Archivo guardado: 'csv/bbva_enriched.csv'
Price      SAN_Price BBVA_Adj_Close BBVA_Close BBVA_Volume BBVA_High  \
Ticker        SAN.MC        BBVA.MC    BBVA.MC     BBVA.MC   BBVA.MC   
Date                                                                   
2003-12-01  7.746234       3.304151   9.895663    25801137  9.895663   
2003-12-02  7.711614       3.291320   9.857233    22303620  9.905270   
2003-12-03  7.763544       3.313776   9.924485    14010666  9.934092   
2003-12-04  7.728924       3.323397   9.953307    19892058  9.972522   
2003-12-05  7.702959       3.291320   9.857233    12585450  9.962915   

Price       BBVA_Low BBVA_Open         IBEX        SP500    EURUS

In [5]:
# leer sin cabecera porque las tres primeras filas son especiales
df_raw = pd.read_csv("../csv/santander_enriched.csv", header=None)

# 1) la primera fila tiene los nombres buenos
column_names = df_raw.iloc[0].tolist()

# 2) nos quedamos con los datos reales (a partir de la fila 3 → índice 3)
df = df_raw.iloc[3:].reset_index(drop=True)

# 3) ponemos los nombres de columna
df.columns = column_names

# 4) renombrar 'Price' -> 'Date' (porque es la fecha en realidad)
df = df.rename(columns={"Price": "Date"})

# 5) convertir Date a datetime
df["Date"] = pd.to_datetime(df["Date"])

# 6) pasar el resto a numérico
for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.head())
print(df.dtypes)

df.to_csv("../csv/santander_enriched.csv")
print("💾 Archivo guardado: 'csv/santander_enriched.csv'")


        Date  SAN_Adj_Close  SAN_Close  SAN_Volume  SAN_High   SAN_Low  \
0 2003-12-01       2.415459   7.746234    42034006  7.772199  7.538513   
1 2003-12-02       2.404664   7.711614   187525101  7.763544  7.633719   
2 2003-12-03       2.420857   7.763544    31459710  7.763544  7.651029   
3 2003-12-04       2.410062   7.728924    37904447  7.746234  7.668339   
4 2003-12-05       2.401966   7.702959    20656590  7.728924  7.651029   

   SAN_Open  BBVA_Price         IBEX        SP500    EURUSD        OIL  
0  7.746234    9.895663  7372.299805  1070.119995  1.196501  29.950001  
1  7.711614    9.857233  7348.700195  1066.619995  1.208897  30.780001  
2  7.763544    9.924485  7384.299805  1064.729980  1.212298  31.100000  
3  7.728924    9.953307  7367.100098  1069.719971  1.208094  31.260000  
4  7.702959    9.857233  7348.100098  1061.500000  1.218695  30.730000  
Date             datetime64[ns]
SAN_Adj_Close           float64
SAN_Close               float64
SAN_Volume           

In [6]:
import pandas as pd

# Cargar el CSV
df = pd.read_csv("../csv/santander_enriched.csv", parse_dates=["Date"])
df.sort_values("Date", inplace=True)

# Crear la columna binaria
df["eventos_negativos"] = 0

# Lista definitiva de eventos negativos que afectan a España
eventos_negativos = [
    ("2000-03-01", "2002-12-31"),  # Burbuja puntocom
    ("2004-03-11", "2004-03-31"),  # Atentados 11M
    ("2008-09-01", "2009-06-30"),  # Crisis financiera global
    ("2010-05-01", "2012-12-31"),  # Crisis deuda europea
    ("2012-06-01", "2013-06-30"),  # Rescate bancario español
    ("2020-02-15", "2021-06-30"),  # COVID-19
    ("2022-02-24", "2023-12-31"),  # Guerra Rusia-Ucrania
]

# Marcar los periodos en la columna
for inicio, fin in eventos_negativos:
    mask = (df["Date"] >= inicio) & (df["Date"] <= fin)
    df.loc[mask, "eventos_negativos"] = 1

# Guardar el CSV actualizado
df.to_csv("../csv/santander_enriched.csv", index=False)

print("✅ Columna 'eventos_negativos' añadida correctamente.")


✅ Columna 'eventos_negativos' añadida correctamente.


In [9]:
# Cargar el CSV
df = pd.read_csv("../csv/bbva_enriched.csv", parse_dates=["Date"])
df.sort_values("Date", inplace=True)

# Crear la columna binaria
df["eventos_negativos"] = 0

# Lista definitiva de eventos negativos que afectan a España
eventos_negativos = [
    ("2000-03-01", "2002-12-31"),  # Burbuja puntocom
    ("2004-03-11", "2004-03-31"),  # Atentados 11M
    ("2008-09-01", "2009-06-30"),  # Crisis financiera global
    ("2010-05-01", "2012-12-31"),  # Crisis deuda europea
    ("2012-06-01", "2013-06-30"),  # Rescate bancario español
    ("2020-02-15", "2021-06-30"),  # COVID-19
    ("2022-02-24", "2023-12-31"),  # Guerra Rusia-Ucrania
]

# Marcar los periodos en la columna
for inicio, fin in eventos_negativos:
    mask = (df["Date"] >= inicio) & (df["Date"] <= fin)
    df.loc[mask, "eventos_negativos"] = 1

# Guardar el CSV actualizado
df.to_csv("../csv/bbva_enriched.csv", index=False)

print("✅ Columna 'eventos_negativos' añadida correctamente.")

✅ Columna 'eventos_negativos' añadida correctamente.


In [8]:
import pandas as pd
import yfinance as yf

# Descargar datos del VIX
vix = yf.download(
    "^VIX",
    start="2000-01-01",
    auto_adjust=False,
    group_by="column"
)

# Solución más directa: aplanar siempre el MultiIndex si existe
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(-1)

# Seleccionar Close y renombrar
vix = vix[["^VIX"]].rename(columns={"^VIX": "vix"})

# Reset index
vix = vix.reset_index()

# Merge
df["Date"] = pd.to_datetime(df["Date"])
vix["Date"] = pd.to_datetime(vix["Date"])
df = df.merge(vix, on="Date", how="left")
df["vix"] = df["vix"].ffill()


# guardar el csv actualizado
df.to_csv("../csv/bbva_enriched.csv", index=False)

[*********************100%***********************]  1 of 1 completed


In [1]:
import pandas as pd
import requests

# 1. bajar del Banco Mundial
url = "https://api.worldbank.org/v2/country/ESP/indicator/NY.GDP.MKTP.KD.ZG"
params = {"format": "json", "per_page": 2000}
data = requests.get(url, params=params).json()

rows = []
for item in data[1]:
    # algunos años vienen con value = None
    if item["value"] is not None:
        rows.append({"date": item["date"], "gdp_growth": item["value"]})

df_gdp = pd.DataFrame(rows)

# 2. pasar a int/año y ordenar de antiguo a reciente
df_gdp["year"] = df_gdp["date"].astype(int)
df_gdp = df_gdp.sort_values("year")  # ahora sí verás los 90s/2000s si los hay

# opcional: ver qué hay
print(df_gdp.tail(15))

# 3. cargar tu df diario
df = pd.read_csv("../csv/santander_enriched.csv", parse_dates=["Date"])

# 4. sacar el año del diario
df["year"] = df["Date"].dt.year

# 5. merge por año
df = df.merge(df_gdp[["year", "gdp_growth"]], on="year", how="left")

# 6. si no quieres dejar el year suelto:
# df = df.drop(columns=["year"])

df.to_csv("../csv/santander_enriched.csv", index=False)



    date  gdp_growth  year
14  2010    0.094122  2010
13  2011   -0.639921  2011
12  2012   -2.865114  2012
11  2013   -1.427322  2013
10  2014    1.520486  2014
9   2015    4.060867  2015
8   2016    2.915156  2016
7   2017    2.896042  2017
6   2018    2.395411  2018
5   2019    1.961179  2019
4   2020  -10.940071  2020
3   2021    6.683144  2021
2   2022    6.179312  2022
1   2023    2.675663  2023
0   2024    3.150196  2024


In [19]:
import pandas as pd
import pandas_datareader.data as web
import datetime as dt

start = dt.datetime(2003, 12, 1)

# Serie: Harmonized Index of Consumer Prices: All-Items HICP for Spain
# código: CP0000ESM086NEST
hicp = web.DataReader("CP0000ESM086NEST", "fred", start)

# limpiar
hicp = hicp.rename(columns={"CP0000ESM086NEST": "hicp_es"})
hicp = hicp.reset_index().rename(columns={"DATE": "Date"})

print(hicp.head())
df = pd.read_csv("../csv/santander_enriched.csv", parse_dates=["Date"])
df = df.merge(hicp, on="Date", how="left")
df.to_csv("../csv/santander_enriched.csv", index=False)


        Date  hicp_es
0 2003-12-01    79.34
1 2004-01-01    78.69
2 2004-02-01    78.76
3 2004-03-01    79.34
4 2004-04-01    80.45


In [30]:
hicp.head()

,Date,hicp_es
0,2003-12-01,79.34
1,2004-01-01,78.69
2,2004-02-01,78.76
3,2004-03-01,79.34
4,2004-04-01,80.45


In [32]:
# asumimos que df ya tiene:
# df["Date"] (diaria)
# df["hicp_es"] (diaria, seguramente ffill del mes)

df["Date"] = pd.to_datetime(df["Date"])

# 1. sacar serie mensual a partir del diario
hicp_monthly = (
    df[["Date", "hicp_es"]]
    .dropna(subset=["hicp_es"])             # por si hay días sin dato
    .set_index("Date")
    .resample("M")                          # fin de mes
    .last()                                 # último valor del mes
)

# 2. inflación interanual: (nivel / nivel hace 12 meses - 1) * 100
hicp_monthly["hicp_es_yoy"] = (
    hicp_monthly["hicp_es"].pct_change(12) * 100
)

# 3. gap con el 2% del BCE
TARGET = 2.0
hicp_monthly["hicp_es_gap_ecb"] = hicp_monthly["hicp_es_yoy"] - TARGET

# 4. volver a diario: reindexar con las fechas diarias de tu df y ffill
#    primero ponemos índice Date también en df
df = df.set_index("Date").sort_index()

# alineamos por índice y rellenamos
df["hicp_es_yoy"] = hicp_monthly["hicp_es_yoy"].reindex(df.index).ffill()
df["hicp_es_gap_ecb"] = hicp_monthly["hicp_es_gap_ecb"].reindex(df.index).ffill()

# 5. si quieres volver a tener Date como columna:
df = df.reset_index()

# guardar el csv actualizado
df.to_csv("../csv/santander_enriched.csv", index=False)


C:\Users\Usuario\AppData\Local\Temp\ipykernel_11360\3972084.py:12: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample("M")                          # fin de mes
C:\Users\Usuario\AppData\Local\Temp\ipykernel_11360\3972084.py:18: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  hicp_monthly["hicp_es"].pct_change(12) * 100


In [20]:
import pandas as pd

# 1. leer el csv del BCE (el que pegaste)
bce_path = "..\\csv\\ECB Data Portal_20251108211527.csv"   # pon aquí la ruta real
df_bce = pd.read_csv(bce_path)

# 2. renombrar columnas a algo manejable
rate_col = "Main refinancing operations - fixed rate tenders (fixed rate) (date of changes) - Level (FM.B.U2.EUR.4F.KR.MRR_FR.LEV)"
df_bce = df_bce.rename(columns={
    "DATE": "date",
    rate_col: "ecb_refi_rate"
})

# 3. pasar fecha a datetime
df_bce["date"] = pd.to_datetime(df_bce["date"])

# 4. tu dataframe principal
df["Date"] = pd.to_datetime(df["Date"])

# 5. hacer el merge por fecha
df = df.merge(
    df_bce[["date", "ecb_refi_rate"]],
    left_on="Date",
    right_on="date",
    how="left"
)

# 6. rellenar hacia adelante porque tu df es diario y el BCE solo pone días de cambio
df = df.sort_values("Date")
df["ecb_refi_rate"] = df["ecb_refi_rate"].ffill()

# 7. ya no necesitamos la columna 'date'
df = df.drop(columns=["date"])

# 8. (opcional) crear la columna 1 si BCE sube tipos
df["ecb_refi_rate_diff"] = df["ecb_refi_rate"].diff()
df["bce_hike"] = (df["ecb_refi_rate_diff"] > 0).astype(int)

# 9. guardar el csv actualizado
df.to_csv("../csv/santander_enriched.csv", index=False)
print("✅ Archivo guardado: 'csv/santander_enriched.csv'")

✅ Archivo guardado: 'csv/santander_enriched.csv'


In [21]:
import pandas as pd

# lee tu csv enorme
hpi = pd.read_csv("../csv/eurostat_prc_hpi_q_es.csv")

# nombre EXACTO de la columna que queremos
col_es = "Quarterly – Total – Quarterly index, 2015=100 – Spain (Eurostat/prc_hpi_q/Q.TOTAL.I15_Q.ES)"

# nos quedamos con period + esa columna
hpi_es = hpi[["period", col_es]].copy()

# renombramos a algo corto
hpi_es = hpi_es.rename(columns={
    "period": "time",
    col_es: "house_price_index_es"
})

# pasar '2005-Q1' a fecha (fin de trimestre)
hpi_es["time"] = pd.PeriodIndex(hpi_es["time"], freq="Q").to_timestamp()

# ahora tu df diario
df["Date"] = pd.to_datetime(df["Date"])

# merge diario ↔ trimestral
df = df.merge(
    hpi_es,
    left_on="Date",
    right_on="time",
    how="left"
)

# rellenar días entre trimestres
df = df.sort_values("Date")
df["house_price_index_es"] = df["house_price_index_es"].ffill()

# limpiar columna auxiliar
df = df.drop(columns=["time"])

# guardar el csv actualizado
df.to_csv("../csv/santander_enriched.csv", index=False)


In [23]:
import pandas as pd
import yfinance as yf

# Descargar datos del VIX
vix = yf.download(
    "^VIX",
    start="2000-01-01",
    auto_adjust=False,
    group_by="column"
)

# Solución más directa: aplanar siempre el MultiIndex si existe
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(-1)

# Seleccionar Close y renombrar
vix = vix[["^VIX"]].rename(columns={"^VIX": "vix"})

# Reset index
vix = vix.reset_index()

# Merge
df["Date"] = pd.to_datetime(df["Date"])
vix["Date"] = pd.to_datetime(vix["Date"])
df = df.merge(vix, on="Date", how="left")
df["vix"] = df["vix"].ffill()


# guardar el csv actualizado
df.to_csv("../csv/santander_enriched.csv", index=False)


[*********************100%***********************]  1 of 1 completed


In [26]:
df = df.sort_values("Date")

# aseguramos que tenemos el tipo relleno
df["ecb_refi_rate"] = df["ecb_refi_rate"].ffill()

# diferencia día a día
df["ecb_refi_rate_diff"] = df["ecb_refi_rate"].diff()

# ahora la dummy:
# 1 si sube, 0 si baja o se mantiene
df["bce_hike"] = (df["ecb_refi_rate_diff"] > 0).astype(int)

df["bce_hike_strict"] = df["ecb_refi_rate_diff"].apply(
    lambda x: 1 if x > 0 else (0 if x < 0 else None)
)

# guardar el csv actualizado
df.to_csv("../csv/santander_enriched.csv", index=False)
